In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.scoutai_common import get_engine, load_master_view, engineer_features, route_predict

METRIC_MAP = {
    "goals": "total_goals", "assists": "total_assists", "appearances": "total_appearances",
    "minutes played": "minutes_per_match", "international caps": "international_caps",
    "international goals": "international_goals", "goals per 90": "goals_per_90",
    "assists per 90": "assists_per_90", "contract remaining": "contract_months_remaining",
}

POSITION_DEFAULT_METRICS = [
    (["goalkeeper", "gk"], "appearances", "contract remaining"),
    (["back", "centre-back", "center-back", "cb", "sweeper"], "appearances", "international caps"),
    (["wing-back", "wingback"], "assists", "appearances"),
    (["defensive midfield", "dm"], "appearances", "contract remaining"),
    (["midfield"], "assists", "appearances"),
    (["winger", "wing"], "goals", "assists"),
    (["forward", "striker", "centre-forward", "center-forward", "cf"], "goals", "goals per 90"),
]
DEFAULT_FALLBACK = ("goals", "assists")

def suggest_defaults_for_position(position):
    pos_lower = position.lower()
    for keywords, primary, secondary in POSITION_DEFAULT_METRICS:
        if any(kw in pos_lower for kw in keywords):
            return primary, secondary
    return DEFAULT_FALLBACK

In [ ]:
def get_club_recommendations(position, max_budget, min_age, max_age, primary_metric, secondary_metric, top_n, df):
    df = df.copy()
    df['search_pos'] = df['position_group'].astype(str).fillna('') + " " + df['sub_position'].astype(str).fillna('')
    df['search_pos'] = df['search_pos'].str.lower().str.replace("-", " ")
    clean_pos_input = position.replace('I', 'i').lower().replace("-", " ").strip()

    pos_match_df = df[df['search_pos'].str.contains(clean_pos_input, case=False, na=False)]
    budget_match_df = pos_match_df[pos_match_df['current_market_value'] <= max_budget]
    filtered_df = budget_match_df[(budget_match_df['age'] >= min_age) & (budget_match_df['age'] <= max_age)].copy()

    if filtered_df.empty:
        print("\n[WARNING] No players found matching your criteria.")
        return None

    for col in [primary_metric, secondary_metric]:
        col_to_use = col if col in filtered_df.columns else 'total_appearances'
        filtered_df[col_to_use] = pd.to_numeric(filtered_df[col_to_use], errors='coerce').fillna(0)
        rank = filtered_df[col_to_use].rank(pct=True)
        filtered_df[f'norm_{col}'] = 1 / (1 + np.exp(-10 * (rank - 0.5)))

    filtered_df['value_gap'] = (filtered_df['predicted_value'] - filtered_df['current_market_value']) / (filtered_df['current_market_value'] + 1)
    filtered_df['abs_value_gap'] = filtered_df['predicted_value'] - filtered_df['current_market_value']

    filtered_df['pct_gap_score'] = filtered_df['value_gap'].rank(pct=True)
    abs_gap_min = filtered_df['abs_value_gap'].min()
    abs_gap_max = filtered_df['abs_value_gap'].max()
    abs_gap_range = abs_gap_max - abs_gap_min
    filtered_df['abs_gap_score'] = ((filtered_df['abs_value_gap'] - abs_gap_min) / abs_gap_range) if abs_gap_range > 0 else 0.0

    BLEND_ABS_WEIGHT = 0.65
    filtered_df['gap_score'] = (1 - BLEND_ABS_WEIGHT) * filtered_df['pct_gap_score'] + BLEND_ABS_WEIGHT * filtered_df['abs_gap_score']

    filtered_df['Custom_Scout_Score'] = (
        (filtered_df[f'norm_{primary_metric}'] * 0.4) +
        (filtered_df[f'norm_{secondary_metric}'] * 0.3) +
        (filtered_df['gap_score'] * 0.3)
    )

    recommendations = filtered_df.sort_values(by=['Custom_Scout_Score'], ascending=False).head(top_n)
    print(f"\n{'='*80}\n SCOUT AI: OPPORTUNITY LIST (Top {len(recommendations)})\n{'='*80}")

    with open("../data/transfer_list.txt", "w", encoding="utf-8") as f:
        for _, player in recommendations.iterrows():
            pos = player['sub_position'] if pd.notna(player['sub_position']) else player['position_group']
            line = (f"{player['player_name']} | {pos} | {player['club_name']} | "
                    f"Value: E{float(player['current_market_value']):,.0f} | "
                    f"Model ({player['model_used']}): E{float(player['predicted_value']):,.0f} | "
                    f"Gap: {float(player['value_gap'])*100:.1f}% | Score: {float(player['Custom_Scout_Score'])*100:.1f}")
            print(line)
            f.write(line + "\n")
    print(f"\n[SYSTEM] {len(recommendations)} players found and saved to 'data/transfer_list.txt'.")
    return recommendations

In [ ]:
def get_valid_int(prompt):
    while True:
        try: return int(input(prompt))
        except ValueError: print("[ERROR] Invalid number.")

def get_valid_float(prompt):
    while True:
        try: return float(input(prompt))
        except ValueError: print("[ERROR] Invalid number.")

def get_valid_metric(prompt, default_key):
    while True:
        user_input = input(f"{prompt} [Press Enter for '{default_key}']: ").strip().lower()
        if user_input == "": return METRIC_MAP[default_key]
        if user_input in METRIC_MAP: return METRIC_MAP[user_input]
        print(f"\n[ERROR] Available options: {', '.join(METRIC_MAP.keys())}\n")

print("[SYSTEM] Connecting to database and loading models...")
engine = get_engine()
df = load_master_view(engine)
df = engineer_features(df)
df = route_predict(df)

In [ ]:
u_pos = input("1. Position (e.g., Right Winger): ").strip()
u_bud = get_valid_float("2. Max Budget in E: ")
u_min = get_valid_int("3. Min Age: ")
u_max = get_valid_int("4. Max Age: ")

suggested_primary, suggested_secondary = suggest_defaults_for_position(u_pos)
print(f"\n[SYSTEM] Suggested metrics for '{u_pos}': primary='{suggested_primary}', secondary='{suggested_secondary}'")
print(f"Metrics: {', '.join(METRIC_MAP.keys())}")
u_m1 = get_valid_metric("5. Primary Metric", suggested_primary)
u_m2 = get_valid_metric("6. Secondary Metric", suggested_secondary)
u_top = get_valid_int("7. Number of players to view: ")

get_club_recommendations(u_pos, u_bud, u_min, u_max, u_m1, u_m2, u_top, df)